# Session 2: Prompt Engineering for Agentic Behaviors (Student Notebook)

## Learning Objectives

By the end of this session, you will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies
5. **Use LangChain prompt templates** and output parsers for reusable, composable prompt workflows

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

Setup complete. OpenAI client initialized.


---
## Demos

Follow along with these demos. Run each cell and observe the outputs carefully.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a **client feedback classification** task — categorizing engagement survey responses as positive, negative, or neutral.

In [2]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past McKinsey engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a McKinsey client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations — the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

Zero-shot: The sentiment of the feedback can be classified as **neutral**. While it acknowledges that the McKinsey team provided solid analysis (a positive aspect), it also notes a shortcoming in the final recommendations (a negative aspect). Overall, the feedback does not convey a strong positive or negative sentiment but rather a balanced perspective.


Few-shot: NEUTRAL


**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

This mirrors McKinsey's structured problem-solving — breaking a problem into steps before jumping to the answer. CoT is especially valuable for market sizing and financial analysis.

In [3]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT — mirroring McKinsey structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

Without CoT:
To calculate the total revenue after the restructuring, we can follow these steps:

1. **Determine the number of stores that will remain after the closures**:
   - Current number of stores: 1,200
   - Percentage of stores to close: 20%
   - Number of stores to close: 20% of 1,200 = 0.20 * 1,200 = 240 stores
   - Remaining stores: 1,200 - 240 = 960 stores

2. **Calculate the new average revenue per store**:
   - Current average revenue per store: $2 million
   - Increase in revenue per store: 15%
   - New average revenue per store: $2 million * (1 + 0.15) = $2 million * 1.15 = $2.3 million

3. **Calculate the total revenue after the restructuring**:
   - Total revenue from remaining stores: 960 stores * $2.3 million per store = $2,208 million

Thus, the total revenue after the restructuring will be **$2,208 million** or **$2.208 billion**.


With CoT:
Let's solve this step by step as outlined.

### Step 1: Calculate how many stores will be closed

**Total stores** = 1,200  

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different McKinsey practice-area personas that approach the same client question from different angles. This is foundational for building specialized agents in multi-agent consulting systems.

In [4]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=200
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")


McKinsey Growth Strategy Lead:
To help your banking client successfully launch a digital-only subsidiary and effectively compete with fintechs, we'll need to consider a comprehensive strategy focused on growth levers, market opportunities, and revenue acceleration. Here’s a structured approach using McKinsey's frameworks, including the Three Horizons of Growth.

### 1. **Market Opportunity Assessment**

#### Competitive Landscape Analysis
- **Identify Fintech Trends:** Evaluate leading fintechs in the market to determine their offerings, target demographics, and unique value propositions. Understand which segments of the market they are catering to (e.g., neobanks, payment services, peer-to-peer lending).
- **Customer Needs Analysis:** Conduct surveys and focus groups to identify unmet needs or pain points among customers of traditional banks and fintechs, like ease of use, transparency, lower fees, or better digital experiences.

#### Target Market Segmentation
- **Demographics:** As

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems for McKinsey workflows, we often need the LLM output to be machine-readable — for example, extracting structured client data from meeting notes or parsing engagement details from unstructured text. JSON mode ensures the response is valid JSON, making downstream processing reliable.

In [5]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

Parsed JSON output:
{
  "name": "Sarah Chen",
  "title": "CFO",
  "company": "Meridian Healthcare",
  "company_revenue": "$3.2B",
  "location": "Chicago",
  "experience_years": 18,
  "previous_employer": "Deloitte",
  "engagement_priorities": [
    "cost optimization",
    "revenue cycle management",
    "evaluating two potential acquisition targets"
  ]
}

Client Executive: Sarah Chen, CFO
Engagement Priorities: cost optimization, revenue cycle management, evaluating two potential acquisition targets


**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Consulting agents need to maintain context across multiple exchanges — remembering the client industry, engagement scope, and prior recommendations. This demo shows how to build a conversation manager that accumulates context, mirroring how a McKinsey engagement progresses through multiple discussions.

In [6]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=300
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn McKinsey engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.
Assistant: Congratulations on winning the new engagement! To effectively scope and plan this project for the mid-size insurance company, we can break it down using the MECE (Mutually Exclusive, Collectively Exhaustive) framework. 

### Engagement Objectives
1. **Understand Current Claims Processing Costs**
   - Identify current cost drivers.
   - Analyze claims volume and processing times.
   - Assess current technology and systems in use.

2. **Benchmarking and Best Practices**
   - Benchmark against industry standards.
   - Identify best practices in claims processing from leading companies.
   - Explore innovative technologies and approaches used in the market.

3. **Cost Reduction Opportunities**
   - Analyze operational inefficiencies.
   - Identify areas for automation and digitalization.
   - Explore process reengineering opportunities.

4. **Implementation a

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

### Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its `ChatPromptTemplate` supports parameterized prompts, and `OutputParser` classes enforce structured output formats (JSON, lists, etc.) — reducing boilerplate and improving reliability.

In [7]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

# Install if needed: pip install langchain langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for McKinsey Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print("Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print("Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: PydanticOutputParser for McKinsey deliverables ---
print("\n" + "=" * 60)
print("PART B: PydanticOutputParser \u2014 McKinsey Engagement Summary")
print("=" * 60)

# Define the output schema using Pydantic
class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: List[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

output_parser = PydanticOutputParser(pydantic_object=EngagementSummary)
format_instructions = output_parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = output_parser.parse(response.content)

print("Parsed structured output:")
print(f"  executive_summary: {parsed_output.executive_summary}")
print(f"  key_findings: {parsed_output.key_findings}")
print(f"  strategic_priority: {parsed_output.strategic_priority}")
print(f"  confidence: {parsed_output.confidence}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain \u2014 Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | output_parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(f"  executive_summary: {result.executive_summary}")
print(f"  key_findings: {result.key_findings}")
print(f"  strategic_priority: {result.strategic_priority}")
print(f"  confidence: {result.confidence}")

/Users/siddharthsharma/Desktop/mckinsey-genAi-3Day/vnev/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


PART A: LangChain ChatPromptTemplate for McKinsey Analysis
Practice: Operations | Topic: semiconductor supply chain
1. **Supply Chain Vulnerabilities and Geopolitical Risks**: The semiconductor industry is highly susceptible to geopolitical tensions, particularly between major players like the U.S. and China. Recent disruptions, such as export controls and trade restrictions, have highlighted the fragility of global supply chains. Strategic Implication: Companies should diversify their supply sources and consider regionalizing production to mitigate risks associated with geopolitical instability, ensuring a more resilient supply chain.

2. **Technological Advancements and Innovation Cycles**: Rapid technological advancements and shorter product life cycles in the semiconductor sector necessitate agile supply chain operations. Companies that invest in advanced manufacturing technologies, such as AI and IoT, can enhance visibility and responsiveness across their supply chains. Strategic 

---
## Tasks

Now it is your turn! Complete the following tasks by filling in the `TODO` sections. Use the demo code above as reference.

### Task 1: Design Effective System Prompts for a McKinsey Industry Research Agent

Create a system prompt for a McKinsey "Industry Research Agent" and test it with a research question about the electric vehicle battery market.

In [8]:
# Task 1: Design Effective System Prompts for a McKinsey Industry Research Agent

# TODO: Create a system prompt for a McKinsey "Industry Research Agent" that:
# 1. Defines the agent's role as a McKinsey industry research specialist
# 2. Specifies how it should approach research questions (MECE breakdown, data-driven reasoning)
# 3. Defines output format (structured with sections: Executive Summary, Key Findings, Strategic Implications, Confidence Level, Data Gaps)
# 4. Includes constraints (acknowledge uncertainty, use McKinsey frameworks, flag assumptions)
#
# Then test it with a research question about the EV battery market.
#
# Hint: Good system prompts are 100-300 words
# Hint: Include specific output formatting instructions
# Hint: Add behavioral guidelines (e.g., "If uncertain, explicitly state your confidence level")
# Hint: Reference McKinsey frameworks (Porter's Five Forces, value chain analysis, etc.)

research_agent_prompt = ""  # YOUR CODE HERE

def ask_research_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

Implement a ReAct-style prompt where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation) — applied to a McKinsey due diligence scenario.

In [9]:
# Task 2: Implement ReAct-Style Prompting (Reason + Act)

# TODO: Implement a ReAct-style prompt that follows the pattern:
# Thought: [hypothesis or reasoning about what data you need next]
# Action: [what action to take]
# Observation: [result of the action]
# ... (repeat)
# Final Answer: [structured recommendation]
#
# Apply this to a McKinsey due diligence scenario.
# The agent should reason through market data, financials, and competitor analysis.
#
# Hint: Define available "actions" in the system prompt:
#   - MarketData[query]: Look up market size, growth rates, and industry data
#   - FinancialAnalysis[expression]: Perform financial calculations (margins, multiples, CAGR)
#   - CompetitorLookup[company]: Research competitor positioning and market share
# Hint: Structure the system prompt with the ReAct format template
# Hint: The LLM simulates the actions since we don't have real tools yet

def react_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# react_agent("Is a $200M acquisition of a mid-size European logistics company a good deal for our private equity client?")

### Task 3: Create a Reusable McKinsey Prompt Template Library

Build a `PromptTemplate` class with variable placeholders, validation, and a library of pre-built McKinsey consulting templates.

In [10]:
# Task 3: Create a Reusable McKinsey Prompt Template Library

# TODO: Build a PromptTemplate class that:
# 1. Stores template strings with {variable} placeholders
# 2. Has a .format() method to fill in variables
# 3. Has a .validate() method to check all variables are provided
# 4. Includes at least 3 pre-built McKinsey consulting templates:
#    - market_sizing_template: TAM estimation for a product category in a geography
#    - executive_briefing_template: CEO briefing on a topic for an industry client
#    - competitive_analysis_template: Competitive landscape for a company in a sector
#
# Hint: Use Python string .format() or f-strings
# Hint: Use re.findall(r'\{(\w+)\}', template) to find placeholders
# Hint: Raise an error if required variables are missing

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        # YOUR CODE HERE
        pass
    
    def format(self, **kwargs):
        # YOUR CODE HERE
        pass
    
    def validate(self, **kwargs):
        # YOUR CODE HERE
        pass

# Create McKinsey consulting template library
# market_sizing_template = PromptTemplate(...)
# executive_briefing_template = PromptTemplate(...)
# competitive_analysis_template = PromptTemplate(...)

### Task 4: Build a McKinsey Analyst Agent with Consulting Tools

Build a complete agentic loop where the LLM acts as a McKinsey analyst, deciding which consulting tool to call, receiving simulated results, and iterating until it has a final recommendation.

In [11]:
# Task 4: Build a McKinsey Analyst Agent with Consulting Tools

# TODO: Build an agent that:
# 1. Has a system prompt describing available McKinsey consulting tools:
#    - market_database: Look up market size, growth rates, and industry trends
#    - financial_model: Run financial calculations (DCF, multiples, CAGR, margin analysis)
#    - benchmarking: Compare client KPIs against industry benchmarks
# 2. Receives a complex consulting question requiring multiple analytical steps
# 3. Uses structured output (JSON) to specify which tool to call at each step:
#    {"thought": "analytical reasoning", "tool": "tool_name", "input": "tool_input"}
# 4. Processes the "tool results" and generates a final recommendation
#
# Hint: Define tools as a list of dicts with name, description, parameters
# Hint: Ask the LLM to respond with JSON: {"thought": "...", "tool": "name", "input": "..."}
# Hint: Use "final_answer" as the tool name when the agent is ready to give its recommendation
# Hint: Simulate tool responses and feed them back into the conversation
# Hint: Add a max-step limit to prevent infinite loops

tools = [
    {"name": "market_database", "description": "Look up market size, growth rates, and industry trends", "parameters": "query (string)"},
    {"name": "financial_model", "description": "Run financial calculations: DCF, multiples, CAGR, margin analysis", "parameters": "expression (string)"},
    {"name": "benchmarking", "description": "Compare client KPIs against industry benchmarks", "parameters": "metric and industry (string)"}
]

class ToolAgent:
    def __init__(self):
        # YOUR CODE HERE
        pass
    
    def simulate_tool(self, tool_name, tool_input):
        # YOUR CODE HERE
        pass
    
    def process(self, question):
        # YOUR CODE HERE
        pass

# Test: "Should our PE client acquire a specialty chemicals company with $80M revenue and 22% EBITDA margins?"

### Task 5: Build a Few-Shot Consulting Deliverable Classifier

Build a few-shot classifier that categorizes McKinsey deliverable descriptions into one of four categories: `STRATEGY_DECK`, `DUE_DILIGENCE`, `MARKET_SIZING`, or `OPERATIONAL_REVIEW`. Provide labeled examples for each category, then test on unseen descriptions.

In [12]:
# Task 5 - Build a Few-Shot Consulting Deliverable Classifier

# TODO: Create a function classify_deliverable(description) that:
#   1. Uses a system message constraining output to one of 4 categories
#   2. Provides 4 few-shot examples (one per category) as user/assistant message pairs
#   3. Appends the new description and classifies it
#
# Categories: STRATEGY_DECK, DUE_DILIGENCE, MARKET_SIZING, OPERATIONAL_REVIEW
#
# Hint: Structure few-shot examples as alternating user/assistant messages
#   e.g., {"role": "user", "content": "A presentation analyzing..."}, {"role": "assistant", "content": "STRATEGY_DECK"}
# Hint: Use temperature=0 for deterministic classification
# Hint: Use max_tokens=10 since the response is just a category name
# Hint: System message should say "Respond with ONLY the category name"

def classify_deliverable(description):
    """Classify a consulting deliverable into a category using few-shot prompting."""
    # YOUR CODE HERE
    pass


# Test with unseen descriptions
# test_cases = [
#     "A board presentation outlining three acquisition targets and an M&A roadmap.",
#     "A warehouse network optimization study benchmarking fulfillment costs.",
#     "An analysis calculating the serviceable addressable market for EV charging in urban Germany.",
# ]
# for desc in test_cases:
#     print(f"Input: {desc[:70]}...")
#     print(f"Category: {classify_deliverable(desc)}\n")

### Task 6: Chain-of-Thought Market Sizing Agent

Build a function that takes a market sizing question and produces two answers: one direct (no CoT) and one using chain-of-thought reasoning with explicit assumptions, calculations, and sensitivity analysis. Compare the outputs to see how CoT improves transparency and verifiability.

In [13]:
# Task 6 - Chain-of-Thought Market Sizing Agent

# TODO: Create a function market_sizing_cot(question) that:
#   1. Gets a DIRECT answer (no CoT) using a simple system prompt
#   2. Gets a COT answer using a system prompt that enforces this structure:
#      - CLARIFY: What exactly are we estimating?
#      - ASSUMPTIONS: List each assumption with a number
#      - CALCULATION: Show arithmetic step by step
#      - ANSWER: Final market size estimate
#      - SENSITIVITY: Which assumptions matter most?
#   3. Returns both answers for comparison
#
# Hint: The direct answer needs a short system prompt ("You are a market sizing expert. Give a concise estimate.")
# Hint: The CoT system prompt should explicitly list the 5 sections above as required output format
# Hint: Use temperature=0 for both calls for reproducibility
# Hint: Use more max_tokens for CoT (e.g., 500) since the reasoning is longer

def market_sizing_cot(question):
    """Estimate a market size with and without chain-of-thought reasoning."""
    # YOUR CODE HERE
    pass


# Test with a market sizing question
# question = "What is the annual market size for premium dog food in the United States?"
# result = market_sizing_cot(question)
# print("DIRECT:\n", result["direct_answer"])
# print("\nCHAIN-OF-THOUGHT:\n", result["cot_answer"])

### Task 7: Extract Structured Data from Client Meeting Notes

Use JSON mode to extract structured data from unstructured client meeting notes. Your function should return a parsed dictionary with attendees, key decisions, action items (with owners and deadlines), and risks — the kind of data that feeds engagement tracking systems.

In [14]:
# Task 7 - Extract Structured Data from Client Meeting Notes

# TODO: Create a function extract_meeting_data(meeting_notes) that:
#   1. Defines a system prompt with the exact JSON schema to extract
#   2. Uses response_format={"type": "json_object"} to guarantee valid JSON
#   3. Parses the response with json.loads()
#   4. Returns the parsed dict
#
# Required fields in the JSON schema:
#   meeting_date, client_name, attendees (list of name/role/org),
#   key_decisions (list), action_items (list of owner/task/deadline),
#   risks_raised (list), next_meeting
#
# Hint: Put the full JSON schema in the system prompt so the model knows exactly what to output
# Hint: Use response_format={"type": "json_object"} as a kwarg in chat.completions.create()
# Hint: Parse with data = json.loads(response.choices[0].message.content)
# Hint: Use temperature=0 for consistent extraction

def extract_meeting_data(meeting_notes):
    """Extract structured data from meeting notes using JSON mode."""
    # YOUR CODE HERE
    pass


# Test with sample meeting notes
# notes = """Kickoff meeting for Project Phoenix with Apex Manufacturing.
# Date: March 15, 2025. Attendees: Sarah Kim (McKinsey EM), Tom Richardson (Apex CEO).
# Decisions: Focus first sprint on procurement. Defer automation to Phase 2.
# Action: Sarah to deliver spend analysis by March 22. Tom to share PO data.
# Risks: supplier relationship damage. Next meeting: March 22."""
# result = extract_meeting_data(notes)
# print(json.dumps(result, indent=2))

### Task 8: Build a Conversation Summarizer for Long Engagements

Extend the `ConversationManager` from Demo 5 with an auto-summarization feature. When the message history exceeds a configurable threshold, the agent should compress older messages into a summary and continue the conversation with summary + recent messages. This prevents context window overflow in long-running consulting engagements.

In [15]:
# Task 8 - Build a Conversation Summarizer for Long Engagements

# TODO: Build a SmartConversationManager class that:
#   1. __init__(system_prompt, max_history): stores messages list and threshold
#   2. _summarize_history(): when triggered, uses an LLM call to compress older messages
#      into a single summary, keeping only the system message + summary + last 2 exchanges
#   3. send(user_message): checks if history exceeds max_history, summarizes if needed,
#      then appends user message, calls LLM, appends response, returns reply
#   4. get_stats(): returns current message count and number of summaries created
#
# Hint: For _summarize_history, format all messages as "ROLE: content" and ask LLM to summarize
# Hint: After summarizing, rebuild self.messages = [system_msg, summary_msg] + last_4_messages
# Hint: Store the summary as {"role": "system", "content": "[CONVERSATION SUMMARY]\n..."}
# Hint: Use max_history=8 for testing so summarization triggers quickly

class SmartConversationManager:
    def __init__(self, system_prompt, max_history=10):
        """Initialize with system prompt and max message history."""
        # YOUR CODE HERE
        pass

    def _summarize_history(self):
        """Compress older messages into a summary."""
        # YOUR CODE HERE
        pass

    def send(self, user_message):
        """Send a message, auto-summarizing if history exceeds threshold."""
        # YOUR CODE HERE
        pass

    def get_stats(self):
        """Return conversation statistics."""
        # YOUR CODE HERE
        pass


# Test with a multi-turn conversation
# agent = SmartConversationManager(
#     system_prompt="You are a McKinsey engagement planning assistant.",
#     max_history=8
# )
# turns = [
#     "New engagement with a telecom company in Brazil. They want to reduce churn.",
#     "What data should we request first?",
#     "Churn is 3.2% monthly, mostly prepaid customers.",
#     "What hypotheses should we test?",
#     "60% cite network quality, 40% cite pricing.",
#     "What should our recommendation deck look like?",
# ]
# for msg in turns:
#     reply = agent.send(msg)
#     print(f"USER: {msg}")
#     print(f"AGENT: {reply[:150]}...\n")
# print(agent.get_stats())

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.